Código para a obter a lista de tickers que já participaram do S&P 500.

Listas Iniciais

In [ ]:
import pandas as pd
import requests

# URL da página da Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

# Ler tabelas
tables = pd.read_html(response.text, header=[0, 1])  # <- pega os dois níveis do cabeçalho

# Empresas atuais
df_current = tables[0][["Symbol", "Security"]].copy()
df_current.columns = ["Ticker", "Security"]
df_current["Status"] = "current"

# Histórico de mudanças (com MultiIndex)
df_changes = tables[1]

# Achatar os nomes de coluna (ex: ("Added","Ticker") -> "Added_Ticker")
df_changes.columns = ['_'.join(col).strip() for col in df_changes.columns.values]

# "Added" -> Ticker e Security
df_added = df_changes[["Added_Ticker", "Added_Security"]].copy()
df_added.columns = ["Ticker", "Security"]
df_added["Status"] = "added"

# "Removed" -> Ticker e Security
df_removed = df_changes[["Removed_Ticker", "Removed_Security"]].copy()
df_removed.columns = ["Ticker", "Security"]
df_removed["Status"] = "removed"

# Concatenar tudo
df_all = pd.concat([df_current, df_added, df_removed], ignore_index=True)

# Remover duplicados
df_all = df_all.dropna(subset=["Ticker"]).drop_duplicates()
df_all = df_all.dropna(subset=["Ticker"]).drop_duplicates(subset=["Ticker"], keep="first")
df_all.loc[df_all["Status"] != "current", "Status"] = "removed"
df_all = df_all.sort_values("Ticker").reset_index(drop=True)
df_all.to_csv("sp500_historical_list.csv", index=False)

print(df_all.head(20))

Filtrar tickers válidos

In [ ]:
import yfinance as yf
import pandas as pd

# Ler CSV
df_all = pd.read_csv("sp500_historical_list.csv")

# Lista de tickers
tickers = df_all["Ticker"].tolist()

# Função para testar se o ticker é válido
def check_ticker(ticker):
    try:
        data = yf.Ticker(ticker).history(period="1d")
        return not data.empty
    except Exception:
        return False

# Verificar todos os tickers
validity = {ticker: check_ticker(ticker) for ticker in tickers}

# Criar DataFrame com resultados
df_validity = pd.DataFrame(list(validity.items()), columns=["Ticker", "Valid"])
print(df_validity.head())

# Filtrar apenas tickers válidos
valid_tickers = df_validity[df_validity["Valid"]]["Ticker"].tolist()

df_validity.to_csv("tickers_validity.csv", index=False)
pd.DataFrame(valid_tickers, columns=["Ticker"]).to_csv("valid_tickers.csv", index=False)

print(f"Número de tickers válidos: {len(valid_tickers)}")

Encontrar informações sobre os tickers inválidos

In [ ]:
import pandas as pd
import yfinance as yf

# Ler CSVs
df_all = pd.read_csv("sp500_historical_list.csv")
df_validity = pd.read_csv("tickers_validity.csv")

# Tickers válidos e inválidos
valid_tickers = df_validity[df_validity["Valid"] == True]["Ticker"].tolist()
invalid_tickers = df_validity[df_validity["Valid"] == False]["Ticker"].tolist()

print(f"Tickers válidos: {len(valid_tickers)}")
print(f"Tickers inválidos: {len(invalid_tickers)}")

# Função para tentar buscar informações de tickers inválidos
def get_info(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        if info and "longName" in info:
            return {"Ticker": ticker, "Found": True, "Name": info.get("longName"), "Exchange": info.get("exchange")}
        else:
            return {"Ticker": ticker, "Found": False, "Name": None, "Exchange": None}
    except Exception:
        return {"Ticker": ticker, "Found": False, "Name": None, "Exchange": None}

# Rodar análise nos inválidos
results = [get_info(ticker) for ticker in invalid_tickers]

# Criar DataFrame com resultados
df_invalid_analysis = pd.DataFrame(results)
df_invalid_analysis.to_csv("tickers_invalid_analysis.csv", index=False)

print(df_invalid_analysis.head())

obter dados dos tickers válidos

In [1]:
import yfinance as yf
import pandas as pd
import time

# Ler lista de tickers válidos
df_tickers = pd.read_csv("valid_tickers.csv")
tickers = df_tickers["Ticker"].tolist()

# DataFrames para armazenar os dados
df_cluster_prices = pd.DataFrame()
df_cluster_dividends = pd.DataFrame()
df_cluster_splits = pd.DataFrame()

df_reg_financials = pd.DataFrame()
df_reg_quarterly_financials = pd.DataFrame()
df_reg_balance = pd.DataFrame()
df_reg_quarterly_balance = pd.DataFrame()
df_reg_cashflow = pd.DataFrame()
df_reg_quarterly_cashflow = pd.DataFrame()

# Função para extrair informações de cluster
def get_cluster_data(ticker):
    try:
        t = yf.Ticker(ticker)

        # Preços históricos diários (cluster)
        hist = t.history(period="max", interval="1d").reset_index()
        hist["Ticker"] = ticker
        global df_cluster_prices
        df_cluster_prices = pd.concat([df_cluster_prices, hist], ignore_index=True)

        # Dividendos
        divs = t.dividends.reset_index()
        divs["Ticker"] = ticker
        global df_cluster_dividends
        df_cluster_dividends = pd.concat([df_cluster_dividends, divs], ignore_index=True)

        # Splits
        splits = t.splits.reset_index()
        splits["Ticker"] = ticker
        global df_cluster_splits
        df_cluster_splits = pd.concat([df_cluster_splits, splits], ignore_index=True)

        print(f"📊 Cluster - {ticker} coletado")
    except Exception as e:
        print(f"❌ Erro cluster em {ticker}: {e}")

# Função para extrair informações para regressão
def get_regression_data(ticker):
    try:
        t = yf.Ticker(ticker)

        # Financeiros anuais
        fin = t.financials
        if not fin.empty:
            fin = fin.T.reset_index().rename(columns={"index": "Date"})
            fin["Ticker"] = ticker
            global df_reg_financials
            df_reg_financials = pd.concat([df_reg_financials, fin], ignore_index=True)

        # Financeiros trimestrais
        qfin = t.quarterly_financials
        if not qfin.empty:
            qfin = qfin.T.reset_index().rename(columns={"index": "Date"})
            qfin["Ticker"] = ticker
            global df_reg_quarterly_financials
            df_reg_quarterly_financials = pd.concat([df_reg_quarterly_financials, qfin], ignore_index=True)

        # Balanço patrimonial anual
        bal = t.balance_sheet
        if not bal.empty:
            bal = bal.T.reset_index().rename(columns={"index": "Date"})
            bal["Ticker"] = ticker
            global df_reg_balance
            df_reg_balance = pd.concat([df_reg_balance, bal], ignore_index=True)

        # Balanço patrimonial trimestral
        qbal = t.quarterly_balance_sheet
        if not qbal.empty:
            qbal = qbal.T.reset_index().rename(columns={"index": "Date"})
            qbal["Ticker"] = ticker
            global df_reg_quarterly_balance
            df_reg_quarterly_balance = pd.concat([df_reg_quarterly_balance, qbal], ignore_index=True)

        # Fluxo de caixa anual
        cf = t.cashflow
        if not cf.empty:
            cf = cf.T.reset_index().rename(columns={"index": "Date"})
            cf["Ticker"] = ticker
            global df_reg_cashflow
            df_reg_cashflow = pd.concat([df_reg_cashflow, cf], ignore_index=True)

        # Fluxo de caixa trimestral
        qcf = t.quarterly_cashflow
        if not qcf.empty:
            qcf = qcf.T.reset_index().rename(columns={"index": "Date"})
            qcf["Ticker"] = ticker
            global df_reg_quarterly_cashflow
            df_reg_quarterly_cashflow = pd.concat([df_reg_quarterly_cashflow, qcf], ignore_index=True)

        print(f"📈 Regressão - {ticker} coletado")
    except Exception as e:
        print(f"❌ Erro regressão em {ticker}: {e}")

# Loop por tickers
for i, ticker in enumerate(tickers):
    get_cluster_data(ticker)
    get_regression_data(ticker)
    time.sleep(0.5)  # delay para evitar bloqueio

    if (i+1) % 20 == 0:
        print(f"{i+1}/{len(tickers)} tickers processados")

# Salvar em CSV
df_cluster_prices.to_csv("cluster_prices.csv", index=False)
df_cluster_dividends.to_csv("cluster_dividends.csv", index=False)
df_cluster_splits.to_csv("cluster_splits.csv", index=False)

df_reg_financials.to_csv("reg_financials.csv", index=False)
df_reg_quarterly_financials.to_csv("reg_quarterly_financials.csv", index=False)
df_reg_balance.to_csv("reg_balance.csv", index=False)
df_reg_quarterly_balance.to_csv("reg_quarterly_balance.csv", index=False)
df_reg_cashflow.to_csv("reg_cashflow.csv", index=False)
df_reg_quarterly_cashflow.to_csv("reg_quarterly_cashflow.csv", index=False)

print("✅ Todos os dados salvos!")

📊 Cluster - A coletado
📈 Regressão - A coletado
📊 Cluster - AA coletado
📈 Regressão - AA coletado
📊 Cluster - AAL coletado
📈 Regressão - AAL coletado
📊 Cluster - AAP coletado
📈 Regressão - AAP coletado
📊 Cluster - AAPL coletado
📈 Regressão - AAPL coletado
📊 Cluster - ABBV coletado
📈 Regressão - ABBV coletado


$ABNB: possibly delisted; no price data found  (1d 1926-09-30 -> 2025-09-05)


📊 Cluster - ABNB coletado
📈 Regressão - ABNB coletado
📊 Cluster - ABT coletado
📈 Regressão - ABT coletado
📊 Cluster - ACGL coletado
📈 Regressão - ACGL coletado
📊 Cluster - ACN coletado
📈 Regressão - ACN coletado
📊 Cluster - ACT coletado
📈 Regressão - ACT coletado
📊 Cluster - ADBE coletado
📈 Regressão - ADBE coletado
📊 Cluster - ADCT coletado
📈 Regressão - ADCT coletado
📊 Cluster - ADI coletado
📈 Regressão - ADI coletado
📊 Cluster - ADM coletado
📈 Regressão - ADM coletado
📊 Cluster - ADP coletado
📈 Regressão - ADP coletado
📊 Cluster - ADSK coletado
📈 Regressão - ADSK coletado
📊 Cluster - ADT coletado
📈 Regressão - ADT coletado
📊 Cluster - AEE coletado
📈 Regressão - AEE coletado
📊 Cluster - AEP coletado
📈 Regressão - AEP coletado
20/635 tickers processados
📊 Cluster - AES coletado
📈 Regressão - AES coletado
📊 Cluster - AFL coletado
📈 Regressão - AFL coletado
📊 Cluster - AIG coletado
📈 Regressão - AIG coletado
📊 Cluster - AIV coletado
📈 Regressão - AIV coletado
📊 Cluster - AIZ coletado
📈 

$APH: possibly delisted; no price data found  (1d 1926-09-30 -> 2025-09-05)
C:\Users\joaop\AppData\Local\Temp\ipykernel_10968\2742271477.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_cluster_prices = pd.concat([df_cluster_prices, hist], ignore_index=True)


📊 Cluster - APH coletado
📈 Regressão - APH coletado
📊 Cluster - APO coletado
📈 Regressão - APO coletado
📊 Cluster - APTV coletado
📈 Regressão - APTV coletado
📊 Cluster - ARE coletado
📈 Regressão - ARE coletado
📊 Cluster - ATI coletado
📈 Regressão - ATI coletado
📊 Cluster - ATO coletado
📈 Regressão - ATO coletado
📊 Cluster - AVB coletado
📈 Regressão - AVB coletado
📊 Cluster - AVGO coletado
📈 Regressão - AVGO coletado
📊 Cluster - AVY coletado
📈 Regressão - AVY coletado
📊 Cluster - AWK coletado
📈 Regressão - AWK coletado
📊 Cluster - AXON coletado
📈 Regressão - AXON coletado
60/635 tickers processados
📊 Cluster - AXP coletado
📈 Regressão - AXP coletado
📊 Cluster - AYI coletado
📈 Regressão - AYI coletado
📊 Cluster - AZO coletado
📈 Regressão - AZO coletado
📊 Cluster - BA coletado
📈 Regressão - BA coletado
📊 Cluster - BAC coletado
📈 Regressão - BAC coletado
📊 Cluster - BALL coletado
📈 Regressão - BALL coletado
📊 Cluster - BAX coletado
📈 Regressão - BAX coletado
📊 Cluster - BBBY coletado
📈 Reg

...

📊 Cluster - BBWI coletado
📈 Regressão - BBWI coletado
📊 Cluster - BBY coletado
📈 Regressão - BBY coletado
📊 Cluster - BC coletado
📈 Regressão - BC coletado
📊 Cluster - BDX coletado
📈 Regressão - BDX coletado
📊 Cluster - BEAM coletado
📈 Regressão - BEAM coletado
📊 Cluster - BEN coletado
📈 Regressão - BEN coletado
📊 Cluster - BG coletado
📈 Regressão - BG coletado
📊 Cluster - BHF coletado
📈 Regressão - BHF coletado
📊 Cluster - BIIB coletado
📈 Regressão - BIIB coletado
📊 Cluster - BIO coletado
📈 Regressão - BIO coletado
📊 Cluster - BK coletado
📈 Regressão - BK coletado
📊 Cluster - BKNG coletado
📈 Regressão - BKNG coletado
80/635 tickers processados
📊 Cluster - BKR coletado
📈 Regressão - BKR coletado
📊 Cluster - BLDR coletado
📈 Regressão - BLDR coletado
📊 Cluster - BLK coletado
📈 Regressão - BLK coletado
📊 Cluster - BMY coletado
📈 Regressão - BMY coletado
📊 Cluster - BR coletado
📈 Regressão - BR coletado
📊 Cluster - BRO coletado
📈 Regressão - BRO coletado
📊 Cluster - BSX coletado
📈 Regressã

$CLF: possibly delisted; no price data found  (1d 1926-09-30 -> 2025-09-05)


📊 Cluster - CLF coletado
📈 Regressão - CLF coletado
📊 Cluster - CLX coletado
📈 Regressão - CLX coletado
📊 Cluster - CMA coletado
📈 Regressão - CMA coletado
120/635 tickers processados
📊 Cluster - CMCSA coletado
📈 Regressão - CMCSA coletado
📊 Cluster - CME coletado
📈 Regressão - CME coletado
📊 Cluster - CMG coletado
📈 Regressão - CMG coletado
📊 Cluster - CMI coletado
📈 Regressão - CMI coletado
📊 Cluster - CMS coletado
📈 Regressão - CMS coletado
📊 Cluster - CNC coletado
📈 Regressão - CNC coletado
📊 Cluster - CNP coletado
📈 Regressão - CNP coletado
📊 Cluster - CNX coletado
📈 Regressão - CNX coletado
📊 Cluster - COF coletado
📈 Regressão - COF coletado
📊 Cluster - COIN coletado
📈 Regressão - COIN coletado
📊 Cluster - COO coletado
📈 Regressão - COO coletado
📊 Cluster - COP coletado
📈 Regressão - COP coletado
📊 Cluster - COR coletado
📈 Regressão - COR coletado
📊 Cluster - COST coletado
📈 Regressão - COST coletado
📊 Cluster - COTY coletado
📈 Regressão - COTY coletado
📊 Cluster - CPAY coletado


$IGT: possibly delisted; no timezone found
C:\Users\joaop\AppData\Local\Temp\ipykernel_10968\2742271477.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_cluster_prices = pd.concat([df_cluster_prices, hist], ignore_index=True)
$IGT: possibly delisted; no timezone found


📊 Cluster - IGT coletado
📈 Regressão - IGT coletado
📊 Cluster - ILMN coletado
📈 Regressão - ILMN coletado
📊 Cluster - INCY coletado
📈 Regressão - INCY coletado
📊 Cluster - INFO coletado
📈 Regressão - INFO coletado
📊 Cluster - INTC coletado
📈 Regressão - INTC coletado
300/635 tickers processados
📊 Cluster - INTU coletado
📈 Regressão - INTU coletado
📊 Cluster - INVH coletado
📈 Regressão - INVH coletado
📊 Cluster - IP coletado
📈 Regressão - IP coletado
📊 Cluster - IPG coletado
📈 Regressão - IPG coletado
📊 Cluster - IPGP coletado
📈 Regressão - IPGP coletado
📊 Cluster - IQV coletado
📈 Regressão - IQV coletado
📊 Cluster - IR coletado
📈 Regressão - IR coletado
📊 Cluster - IRM coletado
📈 Regressão - IRM coletado
📊 Cluster - ISRG coletado
📈 Regressão - ISRG coletado
📊 Cluster - IT coletado
📈 Regressão - IT coletado
📊 Cluster - ITT coletado
📈 Regressão - ITT coletado
📊 Cluster - ITW coletado
📈 Regressão - ITW coletado
📊 Cluster - IVZ coletado
📈 Regressão - IVZ coletado
📊 Cluster - J coletado
📈 R

In [7]:
import nasdaqdatalink as ndl
import pandas as pd
import time

ndl.ApiConfig.api_key = "cn_riEpJs1j5tBynK4bq"

# Ler lista de tickers válidos
df_tickers = pd.read_csv("valid_tickers.csv")
tickers = df_tickers["Ticker"].tolist()

# Apenas para um ticker de teste, para ver todas as colunas
ticker_test = tickers[0]

try:
    df = ndl.get_table(
        'SHARADAR/SF1',
        ticker=ticker_test,
        paginate=True
    )
    print(f"✅ Colunas disponíveis para {ticker_test}:")
    for col in df.columns:
        print(col)
except Exception as e:
    print(f"❌ Erro ao coletar {ticker_test}: {e}")

✅ Colunas disponíveis para A:
ticker
dimension
calendardate
datekey
reportperiod
fiscalperiod
lastupdated
accoci
assets
assetsavg
assetsc
assetsnc
assetturnover
bvps
capex
cashneq
cashnequsd
cor
consolinc
currentratio
de
debt
debtc
debtnc
debtusd
deferredrev
depamor
deposits
divyield
dps
ebit
ebitda
ebitdamargin
ebitdausd
ebitusd
ebt
eps
epsdil
epsusd
equity
equityavg
equityusd
ev
evebit
evebitda
fcf
fcfps
fxusd
gp
grossmargin
intangibles
intexp
invcap
invcapavg
inventory
investments
investmentsc
investmentsnc
liabilities
liabilitiesc
liabilitiesnc
marketcap
ncf
ncfbus
ncfcommon
ncfdebt
ncfdiv
ncff
ncfi
ncfinv
ncfo
ncfx
netinc
netinccmn
netinccmnusd
netincdis
netincnci
netmargin
opex
opinc
payables
payoutratio
pb
pe
pe1
ppnenet
prefdivis
price
ps
ps1
receivables
retearn
revenue
revenueusd
rnd
roa
roe
roic
ros
sbcomp
sgna
sharefactor
sharesbas
shareswa
shareswadil
sps
tangibles
taxassets
taxexp
taxliabilities
tbvps
workingcapital


In [8]:
import nasdaqdatalink as ndl
import pandas as pd
import time

# Configure sua API key
ndl.ApiConfig.api_key = "cn_riEpJs1j5tBynK4bq"

# Ler lista de tickers válidos
df_tickers = pd.read_csv("sp500_historical_list.csv")
tickers = df_tickers["Ticker"].tolist()

# DataFrame final
df_finance = pd.DataFrame()

# Intervalo de datas
start_date = "1900-01-01"
end_date = "2025-01-01"

# Colunas que queremos manter
columns_to_keep = [
    "ticker", "dimension", "calendardate", "revenue", "gp", "ebit", "ebitda", "ebitdamargin",
    "netinc", "netinccmn", "eps", "epsdil", "assets", "assetsavg", "equity", "equityavg",
    "liabilities", "liabilitiesc", "liabilitiesnc", "fcf", "fcfps", "capex", "de", "debt",
    "debtusd", "workingcapital", "tangibles", "pb", "pe", "ps", "marketcap", "dps",
    "divyield", "payoutratio", "roa", "roe", "roic", "sharesbas", "shareswa", "shareswadil"
]

# Limite de chamadas por minuto
CALLS_PER_MINUTE = 10
calls_count = 0
start_time = time.time()

def get_financials_safe(ticker):
    global calls_count, start_time
    try:
        # Respeitar limite de 10 chamadas por minuto
        if calls_count >= CALLS_PER_MINUTE:
            elapsed = time.time() - start_time
            if elapsed < 60:
                wait = 60 - elapsed
                print(f"⏳ Esperando {wait:.1f}s para não exceder limite...")
                time.sleep(wait)
            calls_count = 0
            start_time = time.time()

        df = ndl.get_table(
            'SHARADAR/SF1',
            ticker=ticker,
            calendarDate={'gte': start_date, 'lte': end_date},
            paginate=True
        )
        df = df[columns_to_keep]
        calls_count += 1
        return df

    except Exception as e:
        print(f"❌ Erro ao coletar {ticker}: {e}")
        return pd.DataFrame()

# Loop para todos os tickers
for i, ticker in enumerate(tickers):
    df_temp = get_financials_safe(ticker)
    if not df_temp.empty:
        df_finance = pd.concat([df_finance, df_temp], ignore_index=True)
        print(f"✅ {ticker} coletado ({i+1}/{len(tickers)})")
    else:
        print(f"⚠️ Nenhum dado para {ticker}")
    # Delay extra para segurança (pode reduzir se quiser)
    time.sleep(1)

# Salvar CSV final
df_finance.to_csv("finance_data_regression.csv", index=False)
print("✅ Todos os dados financeiros salvos!")

⚠️ Nenhum dado para A
⚠️ Nenhum dado para AA
⚠️ Nenhum dado para AAL
⚠️ Nenhum dado para AAP
✅ AAPL coletado (5/853)
⚠️ Nenhum dado para ABBV
⚠️ Nenhum dado para ABK
⚠️ Nenhum dado para ABMD
⚠️ Nenhum dado para ABNB
⚠️ Nenhum dado para ABS
⏳ Esperando 36.3s para não exceder limite...
⚠️ Nenhum dado para ABT
⚠️ Nenhum dado para ACAS
⚠️ Nenhum dado para ACE
⚠️ Nenhum dado para ACGL
⚠️ Nenhum dado para ACN
⚠️ Nenhum dado para ACT
⚠️ Nenhum dado para ADBE
⚠️ Nenhum dado para ADCT
⚠️ Nenhum dado para ADI
⚠️ Nenhum dado para ADM
⏳ Esperando 36.7s para não exceder limite...
⚠️ Nenhum dado para ADP
⚠️ Nenhum dado para ADS
⚠️ Nenhum dado para ADSK
⚠️ Nenhum dado para ADT
⚠️ Nenhum dado para AEE
⚠️ Nenhum dado para AEP
⚠️ Nenhum dado para AES
⚠️ Nenhum dado para AET
⚠️ Nenhum dado para AFL
⚠️ Nenhum dado para AGN
⏳ Esperando 33.4s para não exceder limite...
⚠️ Nenhum dado para AIG
⚠️ Nenhum dado para AIV
⚠️ Nenhum dado para AIZ
⚠️ Nenhum dado para AJG
⚠️ Nenhum dado para AKAM
⚠️ Nenhum dado para

Obter Dados Macroeconomicos, indíces e dados de crédito pelo FRED

In [ ]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key='8f01c5703f25d5ea61bb466337b2a5c7')

series = {
    # Macroeconômicos
    "taxa_juros": "FEDFUNDS",
    "inflacao_cpi": "CPIAUCSL",
    "pib_real": "GDPC1",  # trimestral
    "massa_monetaria": "M2SL",
    "desemprego": "UNRATE",
    "dolar_index": "DTWEXBGS",
    "juros_t10y": "GS10",
    "ouro": "PCU2122212122210",

    # Índices de mercado
    "sp500": "SP500",
    "dow_jones": "DJIA",
    "nasdaq": "NASDAQCOM",

    # Crédito
    "corporate_baa": "BAA",
    "credito_cartao": "TERMCBCCALLNS",
    "mortgage_30y": "MORTGAGE30US"
}

df = pd.DataFrame()

for nome, codigo in series.items():
    try:
        serie = fred.get_series(codigo)
        df[nome] = serie
    except Exception as e:
        print(f"Erro ao baixar {nome}: {e}")

# Ajustar frequência mensal (algumas séries podem ser trimestrais ou diárias)
df = df.resample("M").last()  # pega o último valor de cada mês

# Preencher PIB trimestral com forward-fill
df["pib_real"] = df["pib_real"].ffill()

# Salvar CSV
df.to_csv("dados_macro_fred.csv")
print("✅ CSV gerado com sucesso!")

In [2]:
import requests
import pandas as pd
import time

# Identificação obrigatória (troque pelo seu email real!)
headers = {"User-Agent": "joaobussi@usp.br"}

# Baixar lista oficial de tickers/CIKs
url_tickers = "https://www.sec.gov/files/company_tickers.json"
tickers_json = requests.get(url_tickers, headers=headers).json()

# Criar mapa Ticker -> CIK
ticker_to_cik = {entry["ticker"]: str(entry["cik_str"]).zfill(10) for entry in tickers_json.values()}

# Sua lista de tickers
df_tickers = pd.read_csv("sp500_historical_list.csv")
tickers = df_tickers["Ticker"].tolist()

all_data = []

def get_company_facts(cik, ticker, retries=3, delay=2):
    """
    Obtém métricas financeiras de uma empresa pelo CIK no SEC EDGAR.
    Inclui retry automático e delay entre tentativas.
    
    Retorna: lista de dicionários com métricas ou None se não for possível.
    """
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik.zfill(10)}.json"
    
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                data = r.json()
                results = []

                metrics = [
                    "Assets", "Liabilities", "StockholdersEquity",
                    "Revenues", "NetIncomeLoss", "EarningsPerShareDiluted",
                    "NetCashProvidedByUsedInOperatingActivities",
                    "PaymentsOfDividendsCommonStock",
                    "CommonStockSharesOutstanding",
                    # Métricas adicionais
                    "AssetsCurrent", "LiabilitiesCurrent",
                    "CapitalExpenditures", "GrossProfit",
                    "OperatingIncomeLoss", "InterestExpense"
                ]

                for m in metrics:
                    metric_data = data["facts"].get("us-gaap", {}).get(m, {})
                    units_dict = metric_data.get("units", {})
                    
                    # Prioriza USD, depois shares
                    if "USD" in units_dict:
                        vals = units_dict["USD"]
                    elif "shares" in units_dict:
                        vals = units_dict["shares"]
                    else:
                        vals = []

                    for v in vals:
                        results.append({
                            "ticker": ticker,
                            "metric": m,
                            "val": v.get("val"),
                            "end": v.get("end"),
                            "fy": v.get("fy"),
                            "fp": v.get("fp"),
                            "form": v.get("form"),
                            "filed": v.get("filed")
                        })
                return results
            
            else:
                print(f"❌ Erro {r.status_code} para {ticker}")
        
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Tentativa {attempt+1} falhou para {ticker}: {e}")
        
        time.sleep(delay)  # espera antes de tentar novamente
    
    print(f"❌ Não foi possível coletar dados para {ticker} após {retries} tentativas")
    return []

for i, ticker in enumerate(tickers):
    cik = ticker_to_cik.get(ticker)
    if not cik:
        print(f"⚠️ Ticker {ticker} não encontrado")
        continue
    
    data = get_company_facts(cik, ticker)
    if data:
        all_data.extend(data)
        print(f"✅ {ticker} coletado ({i+1}/{len(tickers)})")
    else:
        print(f"⚠️ Nenhum dado para {ticker}")
    
    # Evitar bloqueio da SEC
    time.sleep(0.5)

# Salvar em CSV
df = pd.DataFrame(all_data)
df.to_csv("sec_financials.csv", index=False)
print("📁 Dados salvos em sec_financials.csv")

✅ A coletado (1/853)
✅ AA coletado (2/853)
✅ AAL coletado (3/853)
✅ AAP coletado (4/853)
✅ AAPL coletado (5/853)
✅ ABBV coletado (6/853)
⚠️ Ticker ABK não encontrado
⚠️ Ticker ABMD não encontrado
✅ ABNB coletado (9/853)
⚠️ Ticker ABS não encontrado
✅ ABT coletado (11/853)
⚠️ Ticker ACAS não encontrado
⚠️ Ticker ACE não encontrado
✅ ACGL coletado (14/853)
✅ ACN coletado (15/853)
✅ ACT coletado (16/853)
✅ ADBE coletado (17/853)
✅ ADCT coletado (18/853)
✅ ADI coletado (19/853)
✅ ADM coletado (20/853)
✅ ADP coletado (21/853)
⚠️ Ticker ADS não encontrado
✅ ADSK coletado (23/853)
✅ ADT coletado (24/853)
✅ AEE coletado (25/853)
✅ AEP coletado (26/853)
✅ AES coletado (27/853)
⚠️ Ticker AET não encontrado
✅ AFL coletado (29/853)
⚠️ Ticker AGN não encontrado
✅ AIG coletado (31/853)
✅ AIV coletado (32/853)
✅ AIZ coletado (33/853)
✅ AJG coletado (34/853)
✅ AKAM coletado (35/853)
⚠️ Ticker AKS não encontrado
✅ ALB coletado (37/853)
✅ ALGN coletado (38/853)
✅ ALK coletado (39/853)
✅ ALL coletado (40

In [4]:
import pandas as pd
import yfinance as yf

# ------------------------------
# 1. Carregar dados brutos
# ------------------------------
df = pd.read_csv("sec_financials.csv")
df["end"] = pd.to_datetime(df["end"], errors="coerce")

# ------------------------------
# 2. Pivotar métricas -> colunas
# ------------------------------
df_pivot = df.pivot_table(
    index=["ticker", "end", "fy", "fp", "form", "filed"],
    columns="metric",
    values="val",
    aggfunc="first"
).reset_index()

# ------------------------------
# 3. Separar trimestral e anual
# ------------------------------
df_quarterly = df_pivot[df_pivot["fp"].isin(["Q1", "Q2", "Q3", "Q4"])].copy()
df_annual = df_pivot[df_pivot["fp"].isin(["FY"])].copy()

# ------------------------------
# 4. Adicionar preços históricos
# ------------------------------
def add_prices(df_fin):
    tickers = df_fin["ticker"].unique()
    all_with_prices = []

    for t in tickers:
        df_sub = df_fin[df_fin["ticker"] == t].copy()
        if df_sub.empty:
            continue

        start = df_sub["end"].min() - pd.Timedelta(days=30)
        end = df_sub["end"].max() + pd.Timedelta(days=30)

        try:
            hist = yf.download(
                t,
                start=start.strftime("%Y-%m-%d"),
                end=end.strftime("%Y-%m-%d"),
                progress=False,
                auto_adjust=False  # evita mudanças futuras no padrão
            )

            # Garante coluna única
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.get_level_values(0)

            hist = hist[["Close"]].reset_index()
            hist.rename(columns={"Date": "end", "Close": "ClosePrice"}, inplace=True)

            # Merge aproximado para alinhar datas
            merged = pd.merge_asof(
                df_sub.sort_values("end"),
                hist.sort_values("end"),
                on="end",
                direction="backward"
            )
            all_with_prices.append(merged)
            print(f"💰 Preços adicionados para {t}")
        except Exception as e:
            print(f"⚠️ Erro ao buscar preços para {t}: {e}")

    if all_with_prices:
        return pd.concat(all_with_prices, ignore_index=True)
    else:
        return df_fin

df_quarterly = add_prices(df_quarterly)
df_annual = add_prices(df_annual)

# ------------------------------
# 5. Salvar resultados
# ------------------------------
df_quarterly.to_csv("sec_quarterly.csv", index=False)
df_annual.to_csv("sec_annual.csv", index=False)

print("📁 Dados organizados e salvos: sec_quarterly.csv e sec_annual.csv")

💰 Preços adicionados para A
💰 Preços adicionados para AA
💰 Preços adicionados para AAL
💰 Preços adicionados para AAP
💰 Preços adicionados para AAPL
💰 Preços adicionados para ABBV
💰 Preços adicionados para ABNB
💰 Preços adicionados para ABT
💰 Preços adicionados para ACGL
💰 Preços adicionados para ACN
💰 Preços adicionados para ACT
💰 Preços adicionados para ADBE
💰 Preços adicionados para ADCT
💰 Preços adicionados para ADI
💰 Preços adicionados para ADM
💰 Preços adicionados para ADP
💰 Preços adicionados para ADSK
💰 Preços adicionados para ADT
💰 Preços adicionados para AEE
💰 Preços adicionados para AEP
💰 Preços adicionados para AES
💰 Preços adicionados para AFL
💰 Preços adicionados para AIG
💰 Preços adicionados para AIV
💰 Preços adicionados para AIZ
💰 Preços adicionados para AJG
💰 Preços adicionados para AKAM
💰 Preços adicionados para ALB
💰 Preços adicionados para ALGN
💰 Preços adicionados para ALK
💰 Preços adicionados para ALL
💰 Preços adicionados para ALLE
💰 Preços adicionados para AMAT
💰 


1 Failed download:
['EMN']: Timeout('Failed to perform, curl: (28) Connection timed out after 10013 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


💰 Preços adicionados para EMN
💰 Preços adicionados para EMR
💰 Preços adicionados para ENPH
💰 Preços adicionados para EOG
💰 Preços adicionados para EP
💰 Preços adicionados para EPAM
💰 Preços adicionados para EQIX
💰 Preços adicionados para EQR
💰 Preços adicionados para EQT
💰 Preços adicionados para ERIE
💰 Preços adicionados para ES
💰 Preços adicionados para ESS
💰 Preços adicionados para ETN
💰 Preços adicionados para ETR
💰 Preços adicionados para ETSY
💰 Preços adicionados para EVRG
💰 Preços adicionados para EW
💰 Preços adicionados para EXC
💰 Preços adicionados para EXE
💰 Preços adicionados para EXPD
💰 Preços adicionados para EXPE
💰 Preços adicionados para EXR
💰 Preços adicionados para F
💰 Preços adicionados para FANG
💰 Preços adicionados para FAST
💰 Preços adicionados para FCX
💰 Preços adicionados para FDS
💰 Preços adicionados para FDX
💰 Preços adicionados para FE
💰 Preços adicionados para FFIV
💰 Preços adicionados para FHN
💰 Preços adicionados para FI
💰 Preços adicionados para FICO
💰 Pre